In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import os
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("/kaggle/input/datasets/indranil00/final-movie-data/complete_dataset_merged(1).csv")
df.head(2)

,movieId,clean_title,year,genres_x,avg_rating,num_votes,underrated_score,tag,combined_text,imdbId_x,...,languages,runtime,vote_average,vote_count,release_date,budget,revenue,status,poster_path,backdrop_path
0,1,Toy Story,1995.0,"['Adventure', 'Animation', 'Children', 'Comedy...",3.897438,68997.0,0.349802,"['famous song', 'action', 'arcade', 'national ...",Adventure Animation Children Comedy Fantasy fa...,114709,...,['en'],81.0,7.971,19651.0,1995-11-22,30000000.0,401157969.0,Released,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...,https://image.tmdb.org/t/p/w780/3Rfvhy1Nl6sSGJ...
1,2,Jumanji,1995.0,"['Adventure', 'Children', 'Fantasy']",3.275758,28904.0,0.318909,"['hacksaw', 'screenplay adapted by author', 'c...",Adventure Children Fantasy hacksaw screenplay ...,113497,...,"['en', 'fr']",104.0,7.245,11172.0,1995-12-15,65000000.0,262821940.0,Released,https://image.tmdb.org/t/p/w500/vgpXmVaVyUL7GG...,https://image.tmdb.org/t/p/w780/qSxeCfWUUyht9h...


In [3]:
df = df.drop(columns=["imdbId_x","tmdbId_x","genres_y","tag","avg_rating","num_votes","underrated_score","combined_text"])

In [4]:
df["underrated_score"] = df["vote_average"] / np.log(df["vote_count"] + 1)
df.head(1)

,movieId,clean_title,year,genres_x,imdbId_y,tmdbId_y,overview,tagline,cast,directors,...,runtime,vote_average,vote_count,release_date,budget,revenue,status,poster_path,backdrop_path,underrated_score
0,1,Toy Story,1995.0,"['Adventure', 'Animation', 'Children', 'Comedy...",114709,862.0,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,"['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim...",['John Lasseter'],...,81.0,7.971,19651.0,1995-11-22,30000000.0,401157969.0,Released,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...,https://image.tmdb.org/t/p/w780/3Rfvhy1Nl6sSGJ...,0.806297


In [5]:
import ast

# convert string → list safely
df["cast"] = df["cast"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["genres_x"] = df["genres_x"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["keywords"] = df["keywords"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [7]:
df["text"] = (
    df["overview"].fillna("") + " " +
    df["keywords"].apply(lambda x: " ".join(x[:15])) + " " +  
    df["genres_x"].apply(lambda x: " ".join(x)) + " " 
)

df["cast_text"] = df["cast"].apply(lambda x: " ".join(x[:5]))

df["director_text"] = df["director"].fillna("")

df["rating_norm"] = df["vote_average"] / 10.0
df["popularity_norm"] = np.log(df["vote_count"] + 1)
df["popularity_norm"] /= df["popularity_norm"].max()

In [9]:
all_genres = set()

for g_list in df["genres_x"]:
    all_genres.update(g_list)

all_genres = sorted(list(all_genres))

print(all_genres)

def encode_genre(genres, all_genres):
    return [1 if g in genres else 0 for g in all_genres]

df["genre_vec"] = df["genres_x"].apply(lambda x: encode_genre(x, all_genres))

['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [10]:
os.makedirs("/kaggle/working/embeddings", exist_ok=True)

model = SentenceTransformer("all-MiniLM-L6-v2")
#text_embeddings
text_embeddings = model.encode(
    df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# cast_embeddings
cast_embeddings = model.encode(
    df["cast_text"].tolist(),
    normalize_embeddings=True
)

#director_embeddings
director_embeddings = model.encode(
    df["director_text"].tolist(),
    normalize_embeddings=True
)

np.save("/kaggle/working/embeddings/text_embeddings.npy", text_embeddings)
np.save("/kaggle/working/embeddings/cast_embeddings.npy", cast_embeddings)
np.save("/kaggle/working/embeddings/director_embeddings.npy", director_embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1344 [00:00<?, ?it/s]

In [ ]:
text_embeddings = np.load("/kaggle/working/embeddings/text_embeddings.npy")
cast_embeddings = np.load("/kaggle/working/embeddings/cast_embeddings.npy")
director_embeddings = np.load("/kaggle/working/embeddings/director_embeddings.npy")

In [17]:
def jaccard(a, b):
    a, b = set(a), set(b)
    
    if len(a | b) == 0:
        return 0
    
    return len(a & b) / len(a | b)
    
def get_similar(idx):
    
    text_sim = cosine_similarity(
        text_embeddings[idx].reshape(1, -1),
        text_embeddings
    )[0]
    
    cast_sim = cosine_similarity(
        cast_embeddings[idx].reshape(1, -1),
        cast_embeddings
    )[0]
    
    director_sim = cosine_similarity(
        director_embeddings[idx].reshape(1, -1),
        director_embeddings
    )[0]

    query_genres = df.iloc[idx]["genres_x"]

    genre_scores = df["genres_x"].apply(
        lambda g: jaccard(query_genres, g)
    ).values
        
    final_score = (
        0.5 * text_sim +
        0.2 * cast_sim +
        0.1 * genre_scores +
        0.1 * director_sim +
        0.1 * df["rating_norm"].values
    )
    
    return final_score

In [19]:
def recommend(movie_name, top_k=10):
    
    idx = df[df["clean_title"].str.lower() == movie_name.lower()].index[0]
    
    scores = get_similar(idx)
    
    df["score"] = scores
    
    results = df.sort_values("score", ascending=False)
    
    return results.iloc[1:top_k+1][
        ["clean_title", "vote_average", "genres_x"]
    ]

recommend("inception")

,clean_title,vote_average,genres_x
12162,"Dark Knight, The",8.527,"[Action, Crime, Drama, IMAX]"
9374,"Scanner Darkly, A",6.800,"[Animation, Drama, Mystery, Sci-Fi, Thriller]"
9958,Batman Begins,7.721,"[Action, Crime, IMAX]"
66290,Tenet,7.173,"[Action, Thriller]"
17343,"Dark Knight Rises, The",7.800,"[Action, Adventure, Crime, IMAX]"
4100,Memento,8.176,"[Mystery, Thriller]"
76597,Sound of Sun,7.500,"[Drama, Mystery]"
1505,Face/Off,7.051,"[Action, Crime, Drama, Thriller]"
22036,"Groundstar Conspiracy, The",6.100,"[Action, Crime, Mystery, Romance, Sci-Fi, Thri..."
5313,Minority Report,7.351,"[Action, Crime, Mystery, Sci-Fi, Thriller]"
